In [26]:
from datetime import datetime
import pandas as pd
import json

In [27]:
df = pd.read_csv("data.csv")
df.head()


,id,application_date,contracts
0,2925210.0,2024-02-12 19:22:46.652000+00:00,NaN
1,2925211.0,2024-02-12 19:24:29.135000+00:00,"[{""contract_id"": 522530, ""bank"": ""003"", ""summa..."
2,2925212.0,2024-02-12 19:24:41.493000+00:00,NaN
3,2925213.0,2024-02-12 19:24:29.135000+00:00,"[{""contract_id"": 522530, ""bank"": ""003"", ""summa..."
4,2925214.0,2024-02-12 19:24:56.857000+00:00,NaN


In [28]:
non_null_contract = df['contracts'].dropna().iloc[0]
parsed = json.loads(non_null_contract)

parsed



[{'contract_id': 522530,
  'bank': '003',
  'summa': 500000000,
  'loan_summa': 0,
  'claim_date': '13.02.2020',
  'claim_id': 609965,
  'contract_date': '17.02.2020'},
 {'contract_id': '',
  'bank': '014',
  'summa': '',
  'loan_summa': '',
  'claim_date': '28.08.2020',
  'claim_id': 'F00013731',
  'contract_date': ''},
 {'contract_id': '',
  'bank': '014',
  'summa': '',
  'loan_summa': '',
  'claim_date': '08.10.2020',
  'claim_id': 'F00021301',
  'contract_date': ''},
 {'contract_id': '',
  'bank': '014',
  'summa': '',
  'loan_summa': '',
  'claim_date': '25.11.2020',
  'claim_id': 'F00037907',
  'contract_date': ''},
 {'contract_id': '',
  'bank': '053',
  'summa': '',
  'loan_summa': '',
  'claim_date': '09.12.2020',
  'claim_id': 34852,
  'contract_date': ''},
 {'contract_id': 35163,
  'bank': '053',
  'summa': 510000000,
  'loan_summa': 0,
  'claim_date': '15.12.2020',
  'claim_id': 35163,
  'contract_date': '21.12.2020'},
 {'contract_id': '',
  'bank': '014',
  'summa': '',
 

In [29]:
def extract_features(contract_str):
    if pd.isna(contract_str) or not contract_str.strip():
        return {
            'num_contracts': 0,
            'total_summa': 0,
            'num_loans': 0,
            'first_claim_date': None,
            'unique_banks': 0
        }

    try:
        contracts = json.loads(contract_str)
        valid_contracts = [c for c in contracts if isinstance(c, dict)]

        num_contracts = len(valid_contracts)

        summa_values = [
            int(c['summa']) for c in valid_contracts
            if isinstance(c.get('summa'), (int, float)) or str(c.get('summa')).isdigit()
        ]
        total_summa = sum(summa_values)

        num_loans = sum(
            1 for c in valid_contracts
            if str(c.get('loan_summa')).isdigit() and int(c['loan_summa']) > 0
        )

        date_list = []
        for c in valid_contracts:
            raw_date = c.get('claim_date', '')
            try:
                date_list.append(datetime.strptime(raw_date, "%d.%m.%Y"))
            except:
                continue
        first_claim = min(date_list).date() if date_list else None

        unique_banks = len(set(c['bank'] for c in valid_contracts if c.get('bank')))

        return {
            'num_contracts': num_contracts,
            'total_summa': total_summa,
            'num_loans': num_loans,
            'first_claim_date': first_claim,
            'unique_banks': unique_banks
        }

    except Exception:
        return {
            'num_contracts': 0,
            'total_summa': 0,
            'num_loans': 0,
            'first_claim_date': None,
            'unique_banks': 0
        }


In [30]:
#every row , row by row
features_series = df['contracts'].apply(extract_features)

features_df = pd.DataFrame(features_series.tolist())
result = pd.concat([df[['id']], features_df], axis=1)
result.head()


,id,num_contracts,total_summa,num_loans,first_claim_date,unique_banks
0,2925210.0,0,0,0,None,0
1,2925211.0,82,1510000000,0,2020-02-13,7
2,2925212.0,0,0,0,None,0
3,2925213.0,82,1510000000,0,2020-02-13,7
4,2925214.0,0,0,0,None,0


In [8]:
result.to_csv("contract_features.csv", index=False)


In [ ]:
#avg_summa / last_contract_date / total_loan_summa ---- additional_contract_features

In [9]:
def extract_additional_contract_features(contract_str):
    if pd.isna(contract_str) or not contract_str.strip():
        return {
            'num_contracts': 0,
            'total_summa': 0,
            'num_loans': 0,
            'first_claim_date': None,
            'unique_banks': 0,
            'avg_summa': 0,
            'last_contract_date': None,
            'total_loan_summa': 0
        }

    try:
        contracts = json.loads(contract_str)
        valid = [c for c in contracts if isinstance(c, dict)]

        num = len(valid)

        summa_vals = [
            int(c['summa']) for c in valid
            if isinstance(c.get('summa'), (int, float)) or str(c.get('summa')).isdigit()
        ]
        total = sum(summa_vals)
        avg = total / num if num > 0 else 0

        loan_total = sum(
            int(c['loan_summa']) for c in valid
            if str(c.get('loan_summa')).isdigit()
        )

        loans = sum(
            1 for c in valid
            if str(c.get('loan_summa')).isdigit() and int(c['loan_summa']) > 0
        )

        claims = []
        for c in valid:
            raw = c.get('claim_date', '')
            try:
                claims.append(datetime.strptime(raw, "%d.%m.%Y"))
            except:
                continue
        first_claim = min(claims).date() if claims else None

        last_dates = []
        for c in valid:
            raw = c.get('contract_date', '')
            try:
                last_dates.append(datetime.strptime(raw, "%d.%m.%Y"))
            except:
                continue
        last_contract = max(last_dates).date() if last_dates else None

        banks = len(set(c.get('bank') for c in valid if c.get('bank')))

        return {
            'num_contracts': num,
            'total_summa': total,
            'num_loans': loans,
            'first_claim_date': first_claim,
            'unique_banks': banks,
            'avg_summa': avg,
            'last_contract_date': last_contract,
            'total_loan_summa': loan_total
        }

    except:
        return {
            'num_contracts': 0,
            'total_summa': 0,
            'num_loans': 0,
            'first_claim_date': None,
            'unique_banks': 0,
            'avg_summa': 0,
            'last_contract_date': None,
            'total_loan_summa': 0
        }


In [10]:
features_series = df['contracts'].apply(extract_additional_contract_features)
features_df = pd.DataFrame(features_series.tolist())
result = pd.concat([df[['id']], features_df], axis=1)
result.to_csv("contract_features.csv", index=False)


In [11]:
result[result['num_contracts'] == 0].head()

,id,num_contracts,total_summa,num_loans,first_claim_date,unique_banks,avg_summa,last_contract_date,total_loan_summa
0,2925210.0,0,0,0,None,0,0.0,None,0
2,2925212.0,0,0,0,None,0,0.0,None,0
4,2925214.0,0,0,0,None,0,0.0,None,0
7,2925217.0,0,0,0,None,0,0.0,None,0
9,2925219.0,0,0,0,None,0,0.0,None,0


In [12]:
(result['avg_summa'] - result['total_summa'] / result['num_contracts']).abs().max()


np.float64(0.0)